# Medical Emergency Assistant - Gemma 4 E4B Fine-Tuning

Fine-tune Google's Gemma 4 E4B model with QLoRA for concise, effective medical emergency responses.
Optimized for RTX 5070 Ti (16GB VRAM) and edge deployment.

## Setup

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers>=4.52.0 accelerate>=1.5.0 peft>=0.15.0 trl>=0.18.0 datasets bitsandbytes>=0.45.0 scipy matplotlib

In [ ]:
import torch
import time
import json
import numpy as np
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Path setup
root = Path.cwd()
output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)

# HuggingFace login (needed for Gemma gated model)
from huggingface_hub import login
login()  # Paste your HF token when prompted

## Dataset Preparation

We create a medical emergency triage dataset. Each example has a user describing symptoms/situation
and the assistant providing a concise, structured emergency response.

In [ ]:
SYSTEM_PROMPT = """You are an edge-deployed medical emergency assistant. Provide concise, actionable medical guidance.
For every query, respond with:
1. SEVERITY: (Critical/Urgent/Moderate/Low)
2. IMMEDIATE ACTIONS: (numbered steps, max 5)
3. DO NOT: (critical mistakes to avoid)
4. ESCALATION: (when to seek higher-level care)
Be direct. No disclaimers. Every second counts."""

# Example training data - replace with your full medical emergency dataset
# Format: list of {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
training_examples = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Adult male, 55, clutching chest, sweating profusely, pain radiating to left arm. Started 10 minutes ago."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Call emergency services immediately\n2. Have patient sit upright or in position of comfort\n3. Give 325mg aspirin (chew, not swallow whole) if not allergic\n4. Loosen tight clothing around chest and neck\n5. Monitor breathing and consciousness continuously\n\nDO NOT:\n- Do not let patient walk or exert themselves\n- Do not give water or food\n- Do not leave patient unattended\n\nESCALATION: This is a suspected myocardial infarction. Patient needs emergency transport NOW. Begin CPR if patient becomes unresponsive and stops breathing."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Child, 8 years old, fell from a tree about 6 feet. Crying, holding right forearm which looks bent at an odd angle. No head injury visible."},
            {"role": "assistant", "content": "SEVERITY: Urgent\n\nIMMEDIATE ACTIONS:\n1. Keep the child still and calm\n2. Immobilize the forearm in the position found - do not attempt to straighten\n3. Splint using rigid material (board, rolled magazine) padded with cloth\n4. Apply ice wrapped in cloth to reduce swelling (not directly on skin)\n5. Check fingers for circulation: color, warmth, sensation, movement\n\nDO NOT:\n- Do not try to realign or set the bone\n- Do not remove clothing by pulling over the injury (cut it away)\n- Do not apply pressure directly on the deformity\n\nESCALATION: Transport to emergency facility for X-ray and proper setting. If fingers become blue, cold, or numb, this indicates circulation compromise - expedite transport."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Woman, 30s, having a severe allergic reaction. Lips swelling, difficulty breathing, hives all over body. She was stung by a bee 5 minutes ago."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Use epinephrine auto-injector (EpiPen) on outer thigh immediately if available\n2. Call emergency services\n3. Have patient lie down with legs elevated unless breathing is difficult (then sit upright)\n4. Remove bee stinger by scraping sideways with flat edge (credit card) - do not squeeze\n5. Give second EpiPen dose after 5 minutes if no improvement and EMS not arrived\n\nDO NOT:\n- Do not wait to see if symptoms improve on their own\n- Do not give oral antihistamines as primary treatment (too slow)\n- Do not have patient stand or walk\n\nESCALATION: Anaphylaxis is life-threatening. Even if EpiPen improves symptoms, patient MUST be transported to hospital - biphasic reactions can occur hours later."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Hiker, 40s, confused and stumbling. Hot dry skin, stopped sweating. Temperature feels very high. Been hiking in 100F heat for 4 hours."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Move to shade or coolest available area immediately\n2. Call emergency services\n3. Begin rapid cooling: remove excess clothing, pour water over body, fan aggressively\n4. Apply ice packs to neck, armpits, and groin (major blood vessel areas)\n5. If conscious and able to swallow, give small sips of cool water\n\nDO NOT:\n- Do not give large amounts of water to drink quickly\n- Do not immerse in ice water without monitoring (can cause cardiac arrest)\n- Do not leave unattended - seizures may occur\n\nESCALATION: Heatstroke is immediately life-threatening. Core temperature above 104F causes organ damage. Patient needs IV fluids and hospital-level cooling. If patient loses consciousness, place in recovery position and monitor airway."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Deep laceration on forearm from a machete. Bleeding heavily. Person is alert. We are 2 hours from nearest hospital."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Apply firm, direct pressure with cleanest available cloth - do not remove to check\n2. If blood soaks through, add more material on top and maintain pressure\n3. Elevate the arm above heart level while maintaining pressure\n4. If bleeding does not slow after 10 minutes of direct pressure, apply tourniquet 2-3 inches above wound\n5. Mark time of tourniquet application on patient's skin\n\nDO NOT:\n- Do not remove the initial bandage to look at the wound\n- Do not apply a tourniquet directly over a joint\n- Do not loosen tourniquet once applied\n- Do not attempt to clean a heavily bleeding wound first\n\nESCALATION: Begin transport immediately. Monitor for shock signs: rapid pulse, pale/cool skin, confusion, rapid breathing. Keep patient warm and lying down during transport. A tourniquet must be professionally managed within 2 hours."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Person pulled from water, not breathing, has a pulse. Estimated submersion 2-3 minutes."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Place on firm surface on their back, tilt head back to open airway\n2. Begin rescue breathing: pinch nose, seal mouth, give 1 breath every 5 seconds\n3. If vomiting occurs, roll to side, clear airway, return to back and continue\n4. Call emergency services\n5. Continue rescue breathing until patient breathes on their own or EMS arrives\n\nDO NOT:\n- Do not attempt to drain water from lungs (Heimlich is ineffective for drowning)\n- Do not stop rescue breathing if patient vomits - clear and continue\n- Do not assume recovery even if breathing restarts - secondary drowning risk\n\nESCALATION: If pulse is lost at any point, begin full CPR (30 compressions : 2 breaths). Patient must be transported to hospital even if breathing resumes - pulmonary edema can develop hours later."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Diabetic patient, unresponsive, found with insulin syringe nearby. Skin is cold and clammy."},
            {"role": "assistant", "content": "SEVERITY: Critical\n\nIMMEDIATE ACTIONS:\n1. Call emergency services immediately\n2. Place in recovery position (on their side) to protect airway\n3. If glucagon kit available, inject per instructions (usually into thigh muscle)\n4. If no glucagon but patient has any swallow reflex, rub glucose gel or honey on inside of cheeks\n5. Monitor breathing continuously\n\nDO NOT:\n- Do not give anything by mouth if patient cannot swallow\n- Do not administer insulin - this is likely hypoglycemia, not hyperglycemia\n- Do not leave unattended - seizures are possible\n\nESCALATION: Severe hypoglycemia causes brain damage within minutes. If glucagon is given and patient wakes (usually 10-15 min), give fast-acting sugar immediately followed by complex carbs. Transport to hospital regardless of improvement."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Snakebite on ankle. Two puncture marks visible. Swelling starting. Rural area, unknown snake species."},
            {"role": "assistant", "content": "SEVERITY: Urgent\n\nIMMEDIATE ACTIONS:\n1. Keep patient calm and still - movement increases venom spread\n2. Remove rings, watches, tight clothing near bite before swelling worsens\n3. Immobilize the bitten limb at or below heart level\n4. Mark the edge of swelling with pen and note the time\n5. If possible, photograph the snake from safe distance for identification\n\nDO NOT:\n- Do not cut the wound or attempt to suck out venom\n- Do not apply a tourniquet or ice\n- Do not give aspirin or ibuprofen (increases bleeding)\n- Do not apply electric shock\n\nESCALATION: Transport to nearest facility with antivenom. Track swelling progression (mark every 15 min). Signs requiring fastest possible transport: swelling spreading rapidly, difficulty breathing, vision changes, metallic taste, bleeding from gums/nose."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Person having a seizure. Shaking on the ground, eyes rolled back. Started about 1 minute ago."},
            {"role": "assistant", "content": "SEVERITY: Urgent\n\nIMMEDIATE ACTIONS:\n1. Clear area of hard/sharp objects that could cause injury\n2. Place something soft under the head (jacket, towel)\n3. Time the seizure - note when it started\n4. Turn on their side (recovery position) once shaking stops\n5. Stay with patient and speak calmly when they regain consciousness\n\nDO NOT:\n- Do not restrain or hold the person down\n- Do not put anything in their mouth\n- Do not give water or food until fully alert\n- Do not attempt to move them unless in immediate danger\n\nESCALATION: Call emergency services if: seizure lasts more than 5 minutes, second seizure follows without recovery, patient does not regain consciousness within 10 minutes, patient is injured, or this is their first known seizure. Status epilepticus (>5 min) is life-threatening."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Second-degree burn covering most of the forearm from boiling water spill. Blistering, intense pain. Happened 5 minutes ago."},
            {"role": "assistant", "content": "SEVERITY: Urgent\n\nIMMEDIATE ACTIONS:\n1. Cool the burn under cool (not cold) running water for at least 20 minutes\n2. Remove jewelry and clothing near burn area (unless stuck to skin)\n3. After cooling, cover loosely with clean, non-stick dressing or cling film\n4. Give over-the-counter pain relief (ibuprofen or acetaminophen) if available\n5. Keep patient warm (cool the burn, warm the patient - prevent hypothermia)\n\nDO NOT:\n- Do not pop blisters - they protect against infection\n- Do not apply ice, butter, toothpaste, or any home remedies\n- Do not use adhesive bandages directly on the burn\n- Do not wrap tightly - swelling will occur\n\nESCALATION: Forearm-sized second-degree burns need professional care for infection prevention and wound management. Seek medical attention within 24 hours. Go to emergency if: burn wraps around the limb, pain is unmanageable, or signs of infection develop (increasing redness, pus, fever)."}
        ]
    }
]

print(f"Seed examples: {len(training_examples)}")
print("\n--- Example ---")
print(f"User: {training_examples[0]['messages'][1]['content'][:100]}...")
print(f"Assistant: {training_examples[0]['messages'][2]['content'][:100]}...")

In [ ]:
# Save training data and load as HF Dataset
# TODO: Replace the seed examples above with your full dataset.
# You can also load from a JSONL file:
#   train_dataset = load_dataset("json", data_files="your_medical_train.jsonl")["train"]

train_data = training_examples[:8]
val_data = training_examples[8:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## Load Gemma 4 E4B with 4-bit Quantization

Gemma 4 E4B is a ~4B parameter model. With QLoRA (4-bit quantization), it fits
comfortably in ~10GB VRAM, leaving headroom on the RTX 5070 Ti (16GB).

In [ ]:
MODEL_ID = "google/gemma-3n-E4B-it"

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded: {MODEL_ID}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Test Base Model (Before Fine-Tuning)

In [ ]:
def generate_response(model, tokenizer, prompt, system_prompt=None, max_new_tokens=512):
    """Generate a response using chat template."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
        )
    elapsed = time.time() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    tokens_generated = len(new_tokens)
    print(f"[{tokens_generated} tokens in {elapsed:.1f}s = {tokens_generated/elapsed:.1f} tok/s]")
    return response

In [ ]:
test_prompt = "Adult male, 55, clutching chest, sweating profusely, pain radiating to left arm. Started 10 minutes ago."

print("=== BASE MODEL RESPONSE (before fine-tuning) ===")
base_response = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(base_response)

## Format Dataset for Training

In [ ]:
def format_example(example):
    """Format messages into a single text field using the chat template."""
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

print("--- Formatted example ---")
print(train_dataset["text"][0][:500])

## QLoRA Setup

In [ ]:
# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(output_dir / "gemma4-medical-qlora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    max_seq_length=1024,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Save adapter v1
final_path = output_dir / "gemma4-medical-qlora"
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
print(f"Adapter saved to: {final_path}/{ts}_final_adapter_v1")

## Evaluation v1

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = [x["step"] for x in log_history if "loss" in x]
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_epochs = [x["epoch"] for x in log_history if "eval_loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_steps, train_loss, marker="o", markersize=3)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.grid(True)

ax2.plot(eval_epochs, eval_loss, marker="x", color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Eval Loss")
ax2.set_title("Evaluation Loss")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== FINE-TUNED MODEL RESPONSE (v1 - 3 epochs) ===")
response_v1 = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(response_v1)

In [ ]:
# Evaluate on all test scenarios
test_scenarios = [
    "Person having a seizure, shaking on the ground for 2 minutes.",
    "Deep cut on the leg, heavy bleeding, remote location.",
    "Child choking on food, turning blue, cannot cough.",
    "Person collapsed after marathon, confused, hot skin, not sweating.",
    "Elderly person fell, hip pain, cannot stand or move leg.",
]

print("=== Evaluation on test scenarios ===")
for i, scenario in enumerate(test_scenarios):
    print(f"\n{'='*60}")
    print(f"Scenario {i+1}: {scenario}")
    print(f"{'='*60}")
    resp = generate_response(model, tokenizer, scenario, system_prompt=SYSTEM_PROMPT)
    print(resp)

    # Check response structure
    has_severity = "SEVERITY:" in resp
    has_actions = "IMMEDIATE ACTIONS:" in resp
    has_donot = "DO NOT:" in resp
    has_escalation = "ESCALATION:" in resp
    print(f"\n[Structure check: severity={has_severity}, actions={has_actions}, donot={has_donot}, escalation={has_escalation}]")

## Extended Training (v2 - 6 epochs)

In [ ]:
trainer.args.num_train_epochs = 6
trainer.train()

In [ ]:
# Save adapter v2
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
print(f"Adapter v2 saved to: {final_path}/{ts}_final_adapter_v2")

## Evaluation v2

In [ ]:
# Plot updated losses
log_history = trainer.state.log_history

train_steps = [x["step"] for x in log_history if "loss" in x]
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_epochs = [x["epoch"] for x in log_history if "eval_loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_steps, train_loss, marker="o", markersize=3)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss (6 epochs)")
ax1.grid(True)

ax2.plot(eval_epochs, eval_loss, marker="x", color="orange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Eval Loss")
ax2.set_title("Evaluation Loss (6 epochs)")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== FINE-TUNED MODEL RESPONSE (v2 - 6 epochs) ===")
response_v2 = generate_response(model, tokenizer, test_prompt, system_prompt=SYSTEM_PROMPT)
print(response_v2)

In [ ]:
# Re-evaluate on test scenarios
print("=== Evaluation v2 on test scenarios ===")
for i, scenario in enumerate(test_scenarios):
    print(f"\n{'='*60}")
    print(f"Scenario {i+1}: {scenario}")
    print(f"{'='*60}")
    resp = generate_response(model, tokenizer, scenario, system_prompt=SYSTEM_PROMPT)
    print(resp)

    has_severity = "SEVERITY:" in resp
    has_actions = "IMMEDIATE ACTIONS:" in resp
    has_donot = "DO NOT:" in resp
    has_escalation = "ESCALATION:" in resp
    print(f"\n[Structure check: severity={has_severity}, actions={has_actions}, donot={has_donot}, escalation={has_escalation}]")

## Export for Snapdragon X Elite Deployment

Merge the LoRA adapter back into the base model, then convert for on-device inference
on a Snapdragon X Elite laptop (Hexagon NPU + Adreno GPU + Oryon CPU).

Recommended inference runtimes for Snapdragon X Elite:
- **llama.cpp** (GGUF format) — best general-purpose option, ARM-optimized
- **Qualcomm AI Engine Direct / QNN SDK** — leverages the Hexagon NPU for best perf
- **ONNX Runtime** with QNN execution provider — good middle ground

In [ ]:
from peft import AutoPeftModelForCausalLM

# Reload and merge (need full precision for merging)
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    f"{final_path}/{ts}_final_adapter_v2",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()

# Save merged HF model
merged_path = output_dir / "gemma4-medical-merged"
merged_model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))
print(f"Merged model saved to: {merged_path}")

# --- Conversion options for Snapdragon X Elite ---

print("\n=== Option 1: GGUF via llama.cpp (recommended starting point) ===")
print("On the Snapdragon laptop, llama.cpp runs well on the Oryon CPU (ARM NEON).")
print("Steps:")
print(f"  1. git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp")
print(f"  2. pip install -r requirements.txt")
print(f"  3. python convert_hf_to_gguf.py {merged_path} --outtype q4_K_M --outfile gemma4-medical-q4km.gguf")
print(f"  4. Copy .gguf to Snapdragon laptop")
print(f"  5. ./llama-server -m gemma4-medical-q4km.gguf -c 1024 --port 8080")
print()
print("Q4_K_M gives the best quality/size tradeoff (~2.5GB file, fits easily in X Elite's RAM).")
print()
print("=== Option 2: ONNX Runtime + QNN (Hexagon NPU acceleration) ===")
print("For maximum performance using the dedicated NPU:")
print(f"  1. pip install optimum[exporters]")
print(f"  2. optimum-cli export onnx --model {merged_path} gemma4-medical-onnx/")
print(f"  3. On Snapdragon laptop: pip install onnxruntime-qnn")
print(f"  4. Use QNN execution provider for Hexagon NPU inference")
print()
print("=== Option 3: Qualcomm AI Hub ===")
print("Qualcomm AI Hub can compile and optimize models directly for Snapdragon X Elite.")
print("  1. Upload merged model to aihub.qualcomm.com")
print("  2. Select Snapdragon X Elite as target device")
print("  3. Download optimized model binary")

## Next Steps

### 1. Scale Up the Dataset
The seed examples above are minimal. For production quality:
- Expand to 200-500+ examples covering diverse emergencies
- Include trauma, cardiac, respiratory, neurological, environmental, toxicological scenarios
- Have medical professionals review and validate responses
- Add edge cases: pediatric, geriatric, pregnancy-related emergencies

### 2. Increase LoRA Rank
If the model struggles to learn the structured output format:
- Try `r=32` or `r=64` (monitor VRAM usage)
- Increase `lora_alpha` proportionally

### 3. Target More Modules
For deeper adaptation, add MLP layers:
```python
target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
```

### 4. Snapdragon X Elite Deployment Notes
- **RAM**: X Elite has 16-64GB unified RAM — the Q4_K_M quantized model (~2.5GB) leaves plenty of room
- **llama.cpp on ARM**: Build with `cmake -B build -DLLAMA_NATIVE=ON` to enable Oryon-specific optimizations
- **NPU path**: For best latency, explore QNN SDK to offload to the Hexagon NPU — requires ONNX or QNN model format
- **Benchmark target**: Aim for <1s first-token latency and >20 tok/s generation for usable emergency response times
- **Offline-first**: Package the model + runtime as a self-contained app — no internet dependency